# 8. Algorithmic Fairness & Biological Confounder Analysis

**Goal:** Before concluding our study, we must perform rigorous clinical sanity checks on our models and our dataset. This notebook addresses two critical questions regarding demographic variables (specifically, Gender):

1. **Algorithmic Fairness:** Does our final machine learning model perform equally well for both male and female patients, or does it exhibit demographic bias?
2. **Biological Confounders:** Are the microbiomes of men and women fundamentally different in our cohorts? If gender heavily drives microbial abundance, our disease biomarkers might be artifacts of the dataset's sex distribution rather than true inflammatory signals.

**Methodology:**
* We evaluate our top-performing microbiome-only models (PCA + SVM) on stratified gender subsets.
* We perform a univariate Kruskal-Wallis test ($p < 0.01$) strictly between Men and Women within each specific cohort (Healthy, CD, UC) to isolate the biological impact of sex.

In [11]:
# 1. Imports and Setup
import pandas as pd
import numpy as np
import joblib
import os
import warnings
from scipy.stats import kruskal
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
from IPython.display import display

warnings.filterwarnings("ignore")

print("Libraries loaded.")

# Load Data
meta = pd.read_csv("../data/processed/metadata_final.tsv", sep="\t", index_col="Sample")
genera_clr = pd.read_csv("../data/processed/genera_clr.tsv", sep="\t", index_col=0)
meta = meta.loc[genera_clr.index]

# Clean Gender (Fill missing with the most common to prevent data dropping)
meta["Gender"] = meta["Gender"].fillna(meta["Gender"].mode()[0])

def make_dataset_with_gender(meta, genera, group_a, group_b):
    mask = meta["Study.Group"].isin([group_a, group_b])
    meta_sub = meta[mask]
    X = genera.loc[meta_sub.index]
    y = (meta_sub["Study.Group"] == group_a).astype(int)
    gender = meta_sub["Gender"]
    return X, y, gender

# Generate the datasets
X_uc, y_uc, gender_uc = make_dataset_with_gender(meta, genera_clr, "UC", "nonIBD")
X_cd, y_cd, gender_cd = make_dataset_with_gender(meta, genera_clr, "CD", "nonIBD")

print("Data successfully loaded and grouped by Gender.")

Libraries loaded.
Data successfully loaded and grouped by Gender.


### Part 1: Algorithmic Fairness Check

To ensure clinical viability, a diagnostic model must not be biased against a specific demographic. Here, we load the absolute best performing models from our baseline testing (`final_uc_pca_svm.pkl` and `final_cd_pca_svm.pkl`). 

We will generate predictions for the entire dataset, but we will calculate the Area Under the Curve (AUC) separately for male and female patients to check for performance discrepancies ($\Delta$).

In [12]:
print("==========================================================================")
print("=== PART 1: Comprehensive Algorithmic Fairness Audit ===")
print("==========================================================================\n")

class KruskalSelector(BaseEstimator, TransformerMixin):
    def __init__(self, p_threshold=0.01):
        self.p_threshold = p_threshold
        self.selected_features_ = None
        
    def fit(self, X, y):
        pass # Not needed just for loading and predicting
        
    def transform(self, X):
        X_arr = X.values if isinstance(X, pd.DataFrame) else X
        return X_arr[:, self.selected_indices_]
    
    def get_feature_names_out(self):
        return self.selected_features_

def get_metrics(y_true, y_pred, y_prob):
    """Safely calculates all 5 metrics."""
    try: auc = roc_auc_score(y_true, y_prob)
    except ValueError: auc = np.nan # Handles cases where a gender has only 1 class
        
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    
    return auc, acc, f1, prec, rec

def evaluate_all_metrics_by_gender(models_dict, X, y, gender_series):
    """Evaluates multiple models by gender and returns a formatted DataFrame."""
    male_mask = gender_series == "Male"
    female_mask = gender_series == "Female"
    
    results = []
    
    for model_name, file_name in models_dict.items():
        try:
            model = joblib.load(f"../results/{file_name}")
            
            # Predict for everyone
            y_prob = model.predict_proba(X)[:, 1]
            y_pred = model.predict(X)
            
            # Calculate for Males
            m_auc, m_acc, m_f1, m_prec, m_rec = get_metrics(y[male_mask], y_pred[male_mask], y_prob[male_mask])
            
            # Calculate for Females
            f_auc, f_acc, f_f1, f_prec, f_rec = get_metrics(y[female_mask], y_pred[female_mask], y_prob[female_mask])
            
            # Store results with Deltas
            results.append({
                "Model": model_name,
                "AUC (M | F)": f"{m_auc:.3f} | {f_auc:.3f}",
                "Δ AUC": f"{abs(m_auc - f_auc):.3f}",
                "Acc (M | F)": f"{m_acc:.3f} | {f_acc:.3f}",
                "Δ Acc": f"{abs(m_acc - f_acc):.3f}",
                "F1 (M | F)": f"{m_f1:.3f} | {f_f1:.3f}",
                "Δ F1": f"{abs(m_f1 - f_f1):.3f}",
                "Prec (M | F)": f"{m_prec:.3f} | {f_prec:.3f}",
                "Δ Prec": f"{abs(m_prec - f_prec):.3f}",
                "Rec (M | F)": f"{m_rec:.3f} | {f_rec:.3f}",
                "Δ Rec": f"{abs(m_rec - f_rec):.3f}"
            })
            
        except FileNotFoundError:
            print(f"Skipping {model_name}: Model file not found.")
            
    return pd.DataFrame(results)

# Define the 8 models
uc_models = {
    "KW + SVM": "final_uc_kw_svm.pkl",
    "KW + LogReg": "final_uc_kw_logreg.pkl",
    "PCA + SVM": "final_uc_pca_svm.pkl",
    "PCA + LogReg": "final_uc_pca_logreg.pkl"
}

cd_models = {
    "KW + SVM": "final_cd_kw_svm.pkl",
    "KW + LogReg": "final_cd_kw_logreg.pkl",
    "PCA + SVM": "final_cd_pca_svm.pkl",
    "PCA + LogReg": "final_cd_pca_logreg.pkl"
}

# Generate and display the tables
df_fairness_uc = evaluate_all_metrics_by_gender(uc_models, X_uc, y_uc, gender_uc)
df_fairness_cd = evaluate_all_metrics_by_gender(cd_models, X_cd, y_cd, gender_cd)

print(f"--- ULCERATIVE COLITIS (UC) FAIRNESS AUDIT ---")
print(f"Total Males: {(gender_uc == 'Male').sum()} | Total Females: {(gender_uc == 'Female').sum()}")
display(df_fairness_uc)

print(f"\n--- CROHN'S DISEASE (CD) FAIRNESS AUDIT ---")
print(f"Total Males: {(gender_cd == 'Male').sum()} | Total Females: {(gender_cd == 'Female').sum()}")
display(df_fairness_cd)

# Optional: Save these for the report
df_fairness_uc.to_csv("../results/tables/report_fairness_uc.csv", index=False)
df_fairness_cd.to_csv("../results/tables/report_fairness_cd.csv", index=False)
print("\n✅ Fairness audit tables saved to '../results/tables/'")

=== PART 1: Comprehensive Algorithmic Fairness Audit ===

--- ULCERATIVE COLITIS (UC) FAIRNESS AUDIT ---
Total Males: 27 | Total Females: 29


,Model,AUC (M | F),Δ AUC,Acc (M | F),Δ Acc,F1 (M | F),Δ F1,Prec (M | F),Δ Prec,Rec (M | F),Δ Rec
0,KW + SVM,0.733 | 0.687,0.046,0.593 | 0.586,0.006,0.560 | 0.647,0.087,0.538 | 0.688,0.149,0.583 | 0.611,0.028
1,KW + LogReg,0.889 | 0.985,0.096,0.852 | 0.931,0.079,0.818 | 0.941,0.123,0.900 | 1.000,0.100,0.750 | 0.889,0.139
2,PCA + SVM,0.256 | 0.611,0.356,0.370 | 0.690,0.319,0.320 | 0.727,0.407,0.308 | 0.800,0.492,0.333 | 0.667,0.333
3,PCA + LogReg,0.817 | 0.808,0.009,0.630 | 0.828,0.198,0.615 | 0.848,0.233,0.571 | 0.933,0.362,0.667 | 0.778,0.111



--- CROHN'S DISEASE (CD) FAIRNESS AUDIT ---
Total Males: 41 | Total Females: 34


,Model,AUC (M | F),Δ AUC,Acc (M | F),Δ Acc,F1 (M | F),Δ F1,Prec (M | F),Δ Prec,Rec (M | F),Δ Rec
0,KW + SVM,0.067 | 0.257,0.190,0.366 | 0.324,0.042,0.000 | 0.000,0.000,0.000 | 0.000,0.000,0.000 | 0.000,0.000
1,KW + LogReg,0.941 | 0.779,0.162,0.805 | 0.735,0.070,0.826 | 0.780,0.046,0.950 | 0.889,0.061,0.731 | 0.696,0.035
2,PCA + SVM,0.864 | 0.676,0.188,0.707 | 0.559,0.148,0.714 | 0.615,0.099,0.938 | 0.750,0.188,0.577 | 0.522,0.055
3,PCA + LogReg,0.862 | 0.684,0.178,0.707 | 0.529,0.178,0.727 | 0.579,0.148,0.889 | 0.733,0.156,0.615 | 0.478,0.137



✅ Fairness audit tables saved to '../results/tables/'


### Part 2: Biological Confounder Check

Even if a dataset is perfectly balanced, natural biology can skew machine learning interpretations. If the healthy male gut is fundamentally different from the healthy female gut, models might accidentally identify "male-specific" microbes as "disease-causing" microbes simply due to sample distribution.

To prove our biomarkers are driven by Inflammatory Bowel Disease and not by sex, we deploy a Kruskal-Wallis test ($p < 0.01$). However, instead of comparing Sick vs. Healthy, we compare **Male vs. Female** *within* each isolated clinical group. 

An ideal result is $0$ significantly different microbes, confirming the absence of gender confounding.

In [4]:
print("==========================================================================")
print("=== PART 2: Biological Confounder Check (Male vs Female Microbiomes) ===")
print("==========================================================================\n")

def kw_gender_check(X, gender_series, cohort_name, p_threshold=0.01):
    """Runs a univariate KW test to detect gender-driven microbial differences."""
    group_male = X[gender_series == "Male"]
    group_female = X[gender_series == "Female"]
    
    p_values = []
    for col in X.columns:
        stat, p = kruskal(group_male[col].values, group_female[col].values)
        p_values.append(p)
        
    p_series = pd.Series(p_values, index=X.columns)
    significant_features = p_series[p_series < p_threshold]
    
    print(f"Cohort: {cohort_name}")
    print(f" - Cohort Size: {len(group_male)} Males | {len(group_female)} Females")
    print(f" - Microbes significantly driven by sex (p < {p_threshold}): {len(significant_features)} out of {X.shape[1]}")
    
    if len(significant_features) > 0:
        print(f" - ⚠️ Warning: These {len(significant_features)} microbes may act as confounders.")
        # Print the top 3 biggest offenders
        print("   Top confounders by p-value:")
        for bug, p_val in significant_features.nsmallest(3).items():
            clean_bug = bug.split(";")[-1].strip()
            print(f"     * {clean_bug} (p={p_val:.5f})")
    else:
        print(" - ✅ Conclusion: Safe. No significant gender confounding detected within this cohort.")
    print("-" * 75)

# Isolate the datasets by strict clinical group
df_nonibd = genera_clr[meta["Study.Group"] == "nonIBD"]
df_cd = genera_clr[meta["Study.Group"] == "CD"]
df_uc = genera_clr[meta["Study.Group"] == "UC"]

gender_nonibd = meta[meta["Study.Group"] == "nonIBD"]["Gender"]
gender_cd = meta[meta["Study.Group"] == "CD"]["Gender"]
gender_uc = meta[meta["Study.Group"] == "UC"]["Gender"]

# Execute the sanity checks
kw_gender_check(df_nonibd, gender_nonibd, "Healthy Controls (nonIBD)")
kw_gender_check(df_cd, gender_cd, "Crohn's Disease (CD)")
kw_gender_check(df_uc, gender_uc, "Ulcerative Colitis (UC)")

=== PART 2: Biological Confounder Check (Male vs Female Microbiomes) ===

Cohort: Healthy Controls (nonIBD)
 - Cohort Size: 15 Males | 11 Females
 - Microbes significantly driven by sex (p < 0.01): 47 out of 2761
 - ⚠️ Warning: These 47 microbes may act as confounders.
   Top confounders by p-value:
     * g__SIG333 (p=0.00017)
     * g__Extibacter (p=0.00038)
     * g__RGIG423 (p=0.00068)
---------------------------------------------------------------------------
Cohort: Crohn's Disease (CD)
 - Cohort Size: 26 Males | 23 Females
 - Microbes significantly driven by sex (p < 0.01): 14 out of 2761
 - ⚠️ Warning: These 14 microbes may act as confounders.
   Top confounders by p-value:
     * g__Tetrasphaera (p=0.00323)
     * g__Howiella (p=0.00345)
     * g__QAKS01 (p=0.00345)
---------------------------------------------------------------------------
Cohort: Ulcerative Colitis (UC)
 - Cohort Size: 12 Males | 18 Females
 - Microbes significantly driven by sex (p < 0.01): 24 out of 2761
 

In [13]:
# ==========================================================================
# === PART 3: The Final Check (Cross-Referencing Confounders) ===
# ==========================================================================
import pandas as pd
from scipy.stats import kruskal

print("=== PART 3: Cross-Referencing Biomarkers vs. Gender Confounders ===\n")

def get_gender_confounders(X, gender_series, p_threshold=0.01):
    """Returns a set of microbes that are significantly driven by gender."""
    group_male = X[gender_series == "Male"]
    group_female = X[gender_series == "Female"]
    
    confounders = []
    for col in X.columns:
        stat, p = kruskal(group_male[col].values, group_female[col].values)
        if p < p_threshold:
            clean_name = col.split(";")[-1].strip()
            confounders.append(clean_name)
            
    return set(confounders)

# 1. Generate the lists of gender-driven microbes
confounders_healthy = get_gender_confounders(df_nonibd, gender_nonibd)
confounders_cd = get_gender_confounders(df_cd, gender_cd)
confounders_uc = get_gender_confounders(df_uc, gender_uc)

# Combine them into one master list of all gender-driven bugs
all_gender_confounders = confounders_healthy.union(confounders_cd).union(confounders_uc)

# 2. LOAD YOUR SAVED MASTER BIOMARKERS FROM NOTEBOOK 6
try:
    print("Loading Master Consensus Biomarkers from Notebook 6...")
    uc_df = pd.read_csv("../results/tables/uc_master_consensus.csv")
    cd_df = pd.read_csv("../results/tables/cd_master_consensus.csv")
    
    # Extract the exact names of the bugs into sets
    tier1_uc = set(uc_df['Microbe'].tolist())
    tier1_cd = set(cd_df['Microbe'].tolist())

    # 3. Perform the intersection (Check if any ML bugs are actually Gender bugs)
    overlap_uc = tier1_uc.intersection(all_gender_confounders)
    overlap_cd = tier1_cd.intersection(all_gender_confounders)

    print("\n🔍 CHECKING ULCERATIVE COLITIS (UC) BIOMARKERS...")
    if len(overlap_uc) == 0:
        print(" ✅ SUCCESS: 0 overlaps! Your UC biomarkers are 100% pure disease signals.")
    else:
        print(f" ⚠️ WARNING: {len(overlap_uc)} UC biomarker(s) are confounded by gender:")
        for bug in overlap_uc: print(f"    - {bug}")

    print("\n🔍 CHECKING CROHN'S DISEASE (CD) BIOMARKERS...")
    if len(overlap_cd) == 0:
        print(" ✅ SUCCESS: 0 overlaps! Your CD biomarkers are 100% pure disease signals.")
    else:
        print(f" ⚠️ WARNING: {len(overlap_cd)} CD biomarker(s) are confounded by gender:")
        for bug in overlap_cd: print(f"    - {bug}")

except FileNotFoundError:
    print("\n❌ Error: Could not find the CSV files in '../results/tables/'.")
    print("Please make sure you ran Step 1 (the save cell) at the end of Notebook 6!")

=== PART 3: Cross-Referencing Biomarkers vs. Gender Confounders ===

Loading Master Consensus Biomarkers from Notebook 6...

🔍 CHECKING ULCERATIVE COLITIS (UC) BIOMARKERS...
 ✅ SUCCESS: 0 overlaps! Your UC biomarkers are 100% pure disease signals.

🔍 CHECKING CROHN'S DISEASE (CD) BIOMARKERS...
 ✅ SUCCESS: 0 overlaps! Your CD biomarkers are 100% pure disease signals.
